# CP2 — Feature Engineering
**Thesis:** Predicting F1 Race Finishing Positions (2022–2025 ground-effect era), ensemble ML + SHAP.

Checkpoint 2 of 5. Collapses the CP1 lap-level cleaned data to **one row per driver per race**
(~1,826 rows) and builds the model-ready feature matrix. **No model training here.**

**Input:** `data/raw/raw_f1_data_2022_2025_cleaned.csv` (lap-level)
**Outputs:** `data/processed/{race_features,train,test}.csv`, `models/feature_list.json`, `models/encoders.pkl`

### Notes on deviations from the base spec
- **FIX (data leakage):** constructor rolling features are computed at the **team-race level** then
  merged back — not row-wise. Each team has 2 drivers per race, so a row-wise `shift(1)` would leak a
  teammate's *same-race* result. See Step 5.
- **3 extra features auto-added** from an empirical discovery pass (corr-to-target, leakage-checked):
  `clean_pace_delta` (ρ=+0.74), `teammate_pace_delta` (ρ=+0.33), `first_stint_len` (ρ=−0.21).
  Tested-but-rejected (≈0 correlation): tyre-degradation slope, lap-time consistency, lap-1 start gain.
- **Leakage framing:** this is a *race-reconstruction* model — current-race lap-derived pace/strategy/
  condition features are intended; only `final_position` (target), `points_scored`, `status`, and any
  finish-equivalent quantity are excluded. Rolling "form" features use `shift(1)` (strictly pre-race).

## Setup

In [19]:
from pathlib import Path
import json, pickle
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Opt in to future pandas behaviour so ffill/bfill on object (string) columns
# does not emit the silent-downcasting FutureWarning.
pd.set_option("future.no_silent_downcasting", True)

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
CLEAN_CSV = ROOT / "data" / "raw" / "raw_f1_data_2022_2025_cleaned.csv"
PROC_DIR = ROOT / "data" / "processed"; PROC_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = ROOT / "models"; MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(CLEAN_CSV, low_memory=False)
df = df.sort_values(["season", "round", "driver", "lap_number"]).reset_index(drop=True)
KEY = ["season", "round", "driver"]
print("Loaded lap-level:", df.shape)
df.head(3)

Loaded lap-level: (101290, 45)


,season,round,event_name,country,location,date,is_street_circuit,total_race_laps,driver,driver_number,...,is_yellow_flag,is_safety_car,is_vsc,is_red_flag,air_temp,track_temp,humidity,rainfall,is_wet_race,is_accurate
0,2022,1,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,ALB,23,...,False,False,False,False,23.9,29.0,26.0,False,False,False
1,2022,1,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,ALB,23,...,False,False,False,False,23.8,29.0,27.0,False,False,True
2,2022,1,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,ALB,23,...,False,False,False,False,23.8,28.7,29.0,False,False,True


## Step 0 — Team Name Normalisation
Renamed constructors are unified so form features are continuous across seasons.

In [20]:
TEAM_NAME_MAP = {
    "AlphaTauri": "RB", "RB": "RB", "Racing Bulls": "RB",
    "Alfa Romeo": "Kick Sauber", "Kick Sauber": "Kick Sauber",
}
df["team"] = df["team"].replace(TEAM_NAME_MAP)
print(df["team"].value_counts())

team
McLaren            10622
Mercedes           10538
Red Bull Racing    10409
Ferrari            10152
Haas F1 Team       10141
Aston Martin       10127
RB                 10053
Kick Sauber         9904
Alpine              9867
Williams            9477
Name: count, dtype: int64


## Step 1 — Street-Circuit Recompute
(Idempotent — already correct from CP1; re-applied for a self-contained pipeline.)

In [21]:
STREET_CIRCUITS = {
    "Monaco Grand Prix", "Las Vegas Grand Prix", "Abu Dhabi Grand Prix",
    "Australian Grand Prix", "Azerbaijan Grand Prix", "Miami Grand Prix",
    "Singapore Grand Prix", "Saudi Arabian Grand Prix",
}
df["is_street_circuit"] = df["event_name"].isin(STREET_CIRCUITS)
print("Street-circuit laps:", int(df["is_street_circuit"].sum()),
      "| events:", sorted(df.loc[df.is_street_circuit, "event_name"].unique().tolist()))

Street-circuit laps: 32670 | events: ['Abu Dhabi Grand Prix', 'Australian Grand Prix', 'Azerbaijan Grand Prix', 'Las Vegas Grand Prix', 'Miami Grand Prix', 'Monaco Grand Prix', 'Saudi Arabian Grand Prix', 'Singapore Grand Prix']


## Step 2 — Tire Compound Null Fill
Two-stage fill. **Stage 1** forward/back-fills within each driver-race-stint (the intended scope).
For this data that fills 0 of the 780 nulls, because every null is either on a lap with a **NaN
`stint`** (354 laps, excluded from the group) or in a stint where the **entire stint has no compound**
(426 laps) — so there is nothing to pull from within the stint. **Stage 2** therefore falls back to a
driver-race-level fill: every affected driver-race has a known compound on other laps, so the nearest
known compound is carried onto those anomalous laps (typically SC / in/out laps with missing tyre
data). This clears all 780.

In [22]:
before = int(df["tire_compound"].isna().sum())

# Stage 1: fill within driver-race-stint (intended scope).
df["tire_compound"] = (df.groupby(["season", "round", "driver", "stint"])["tire_compound"]
                         .transform(lambda x: x.ffill().bfill()))
after_stint = int(df["tire_compound"].isna().sum())

# Stage 2: fallback fill within the driver-race for laps with NaN stint or whole-stint-null
# compound (anomalous laps). Carries the nearest known compound from the same race.
df["tire_compound"] = (df.groupby(["season", "round", "driver"])["tire_compound"]
                         .transform(lambda x: x.ffill().bfill()))
after_race = int(df["tire_compound"].isna().sum())

print(f"tire_compound nulls: before={before}  after stage1(stint)={after_stint}  "
      f"after stage2(driver-race)={after_race}")

tire_compound nulls: before=780  after stage1(stint)=780  after stage2(driver-race)=0


## Step 3 — Stint Lap Engineering
`tire_age` is inflated by practice laps. `stint_lap` = laps on this tyre since fitted *in the race*.

In [23]:
df["stint_lap"] = (df.groupby(["season", "round", "driver", "stint"])["lap_number"]
                     .transform(lambda x: x - x.min() + 1))
print("stint_lap range:", df["stint_lap"].min(), "->", df["stint_lap"].max(),
      "| stint nulls (laps):", int(df["stint"].isna().sum()))
df[["season", "round", "driver", "lap_number", "stint", "stint_lap"]].head(3)

stint_lap range: 1.0 -> 77.0 | stint nulls (laps): 354


,season,round,driver,lap_number,stint,stint_lap
0,2022,1,ALB,1.0,1.0,1.0
1,2022,1,ALB,2.0,1.0,2.0
2,2022,1,ALB,3.0,1.0,3.0


## Step 4 — Aggregate to Race Level
One row per (season, round, driver). Carry-over columns + strategy + conditions + pace, plus the
auto-added pace/strategy features.

In [24]:
# --- 4a carry-over (constant within a driver-race) ---
carry = ["event_name", "country", "location", "date", "is_street_circuit",
         "total_race_laps", "driver_number", "driver_abbrev", "team",
         "grid_position", "final_position", "points_scored", "laps_completed",
         "status", "dnf", "is_wet_race"]
race_df = df.groupby(KEY, as_index=False).agg({c: "first" for c in carry})

# Pit-lane starts: grid_position==0 means starting from the pit lane (worst slot, behind P20).
# Remap 0 -> 21 so the feature stays monotonic for tree models (further back = larger number);
# leaving it at 0 would falsely imply the pit-lane start is 'better than pole'.
n_pit_lane = int((race_df["grid_position"] == 0).sum())
race_df["grid_position"] = race_df["grid_position"].replace(0, 21)
print(f"Pit-lane starts remapped (grid_position 0 -> 21): {n_pit_lane} driver-races")

# --- 4b strategy ---
race_df["num_pit_stops"] = df.groupby(KEY)["is_pit_lap"].sum().values  # per spec (is_pit_lap count)
first_stint_compound = (df[df["lap_number"] == 1].groupby(KEY)["tire_compound"].first()
                          .rename("first_stint_compound"))

def get_strategy(g):
    stints = g.sort_values("lap_number").groupby("stint")["tire_compound"].first()
    return "→".join(stints.dropna().astype(str).values)
tire_strategy = (df.groupby(KEY).apply(get_strategy, include_groups=False)
                   .rename("tire_strategy"))

# first_stint_len: laps in the opening stint (auto-added feature, rho=-0.21)
min_stint = df.groupby(KEY)["stint"].transform("min")
first_stint_len = (df[df["stint"] == min_stint].groupby(KEY)["stint_lap"].max()
                     .rename("first_stint_len"))

# --- 4c race conditions ---
cond = df.groupby(KEY).agg(
    safety_car_deployed=("is_safety_car", "any"),
    vsc_deployed=("is_vsc", "any"),
    red_flag=("is_red_flag", "any"),
    avg_air_temp=("air_temp", "mean"),
    avg_track_temp=("track_temp", "mean"),
)

# --- 4d pace (accurate laps only) ---
acc = df[(df["is_accurate"] == True) & df["lap_time_seconds"].notna() & (df["lap_time_seconds"] <= 300)]
median_lap_time = acc.groupby(KEY)["lap_time_seconds"].median().rename("median_lap_time")
race_median = acc.groupby(["season", "round"])["lap_time_seconds"].median().rename("race_median")

# clean pace = median of each unit's fastest 20% accurate laps (auto-added, rho=+0.74)
def fastest20_med(s):
    return s.nsmallest(max(1, int(len(s) * 0.2))).median()
clean_pace = acc.groupby(KEY)["lap_time_seconds"].apply(fastest20_med).rename("clean_pace")
race_clean = acc.groupby(["season", "round"])["lap_time_seconds"].apply(fastest20_med).rename("race_clean")

# Merge the pieces onto race_df.
race_df = (race_df
    .merge(first_stint_compound, on=KEY, how="left")
    .merge(tire_strategy, on=KEY, how="left")
    .merge(first_stint_len, on=KEY, how="left")
    .merge(cond, on=KEY, how="left")
    .merge(median_lap_time, on=KEY, how="left")
    .merge(clean_pace, on=KEY, how="left"))
race_df = race_df.merge(race_median, on=["season", "round"], how="left")
race_df = race_df.merge(race_clean, on=["season", "round"], how="left")

# pace deltas (negative = faster than the field)
race_df["pace_delta"] = race_df["median_lap_time"] - race_df["race_median"]
race_df["clean_pace_delta"] = race_df["clean_pace"] - race_df["race_clean"]

print("Race-level shape:", race_df.shape)
race_df.head(3)

Pit-lane starts remapped (grid_position 0 -> 21): 13 driver-races
Race-level shape: (1826, 34)


,season,round,driver,event_name,country,location,date,is_street_circuit,total_race_laps,driver_number,...,vsc_deployed,red_flag,avg_air_temp,avg_track_temp,median_lap_time,clean_pace,race_median,race_clean,pace_delta,clean_pace_delta
0,2022,1,ALB,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,23,...,False,False,22.887719,27.322807,100.7360,98.826,99.32,97.1255,1.4160,1.7005
1,2022,1,ALO,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,14,...,False,False,22.896491,27.322807,99.7445,97.590,99.32,97.1255,0.4245,0.4645
2,2022,1,BOT,Bahrain Grand Prix,Bahrain,Sakhir,2022-03-20,False,57.0,77,...,False,False,22.896491,27.324561,99.6945,97.067,99.32,97.1255,0.3745,-0.0585


In [25]:
# teammate_pace_delta: driver median lap time minus same-race teammate's (auto-added, rho=+0.33).
# Computed on race_df where median_lap_time already lives. 1-driver team-races -> no teammate -> 0 (neutral).
gt = race_df.groupby(["season", "round", "team"])["median_lap_time"]
tsum, tcnt = gt.transform("sum"), gt.transform("count")
race_df["teammate_pace_delta"] = np.where(
    tcnt == 2, race_df["median_lap_time"] - (tsum - race_df["median_lap_time"]), np.nan)
n_no_teammate = int(race_df["teammate_pace_delta"].isna().sum())
race_df["teammate_pace_delta"] = race_df["teammate_pace_delta"].fillna(0.0)
print(f"teammate_pace_delta: {n_no_teammate} driver-races had no valid teammate -> filled 0.0")
print("Race-level shape:", race_df.shape)

teammate_pace_delta: 110 driver-races had no valid teammate -> filled 0.0
Race-level shape: (1826, 35)


## Step 5 — Rolling Form Features (pre-race only, `shift(1)`)
Driver features are row-safe (one row per driver-race). **Constructor features are built at the
team-race level first** to avoid the teammate same-race leakage, then merged back.

In [26]:
race_df = race_df.sort_values(["season", "round"]).reset_index(drop=True)

# --- driver form ---
race_df["driver_rolling_avg_3"] = (race_df.groupby("driver")["final_position"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()))
race_df["driver_rolling_avg_5"] = (race_df.groupby("driver")["final_position"]
    .transform(lambda x: x.shift(1).rolling(5, min_periods=1).mean()))
race_df["driver_dnf_rate_season"] = (race_df.groupby(["season", "driver"])["dnf"]
    .transform(lambda x: x.shift(1).expanding().mean()))

# --- constructor form (LEAKAGE FIX: aggregate to team-race, then merge back) ---
team_race = (race_df.groupby(["season", "round", "team"], as_index=False)
                    .agg(team_race_points=("points_scored", "sum"))
                    .sort_values(["season", "round"]))
team_race["constructor_rolling_avg_points_3"] = (team_race.groupby("team")["team_race_points"]
    .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()))
team_race["constructor_season_points"] = (team_race.groupby(["season", "team"])["team_race_points"]
    .transform(lambda x: x.shift(1).expanding().sum()))
race_df = race_df.merge(
    team_race[["season", "round", "team",
               "constructor_rolling_avg_points_3", "constructor_season_points"]],
    on=["season", "round", "team"], how="left")
print("After rolling features:", race_df.shape)
race_df[["season","round","driver","team","driver_rolling_avg_3",
         "constructor_rolling_avg_points_3","constructor_season_points"]].head(3)

After rolling features: (1826, 40)


,season,round,driver,team,driver_rolling_avg_3,constructor_rolling_avg_points_3,constructor_season_points
0,2022,1,ALB,Williams,NaN,NaN,NaN
1,2022,1,ALO,Alpine,NaN,NaN,NaN
2,2022,1,BOT,Kick Sauber,NaN,NaN,NaN


## Step 6 — Encode Categorical Features

In [27]:
COMPOUND_ORDER = {"SOFT": 0, "MEDIUM": 1, "HARD": 2, "INTERMEDIATE": 3, "WET": 4}
race_df["compound_encoded"] = race_df["first_stint_compound"].map(COMPOUND_ORDER)
# Robustness: fill any residual null with the per-race modal compound code.
race_mode = race_df.groupby(["season", "round"])["compound_encoded"].transform(
    lambda x: x.mode().iloc[0] if not x.mode().empty else 1)
n_filled = int(race_df["compound_encoded"].isna().sum())
race_df["compound_encoded"] = race_df["compound_encoded"].fillna(race_mode).astype(int)
print(f"compound_encoded: filled {n_filled} null(s) with per-race modal compound")

team_encoder = LabelEncoder(); race_df["team_encoded"] = team_encoder.fit_transform(race_df["team"])
driver_encoder = LabelEncoder(); race_df["driver_encoded"] = driver_encoder.fit_transform(race_df["driver"])

with open(MODELS_DIR / "encoders.pkl", "wb") as f:
    pickle.dump({"team": team_encoder, "driver": driver_encoder,
                 "compound_order": COMPOUND_ORDER}, f)
print("Saved models/encoders.pkl |", len(team_encoder.classes_), "teams,",
      len(driver_encoder.classes_), "drivers")

compound_encoded: filled 0 null(s) with per-race modal compound
Saved models/encoders.pkl | 10 teams, 31 drivers


## Step 7 — Final Feature Set
Base 21 features + 3 auto-added (`clean_pace_delta`, `teammate_pace_delta`, `first_stint_len`) = 24.

In [28]:
FEATURE_COLS = [
    # Starting conditions
    "grid_position", "is_street_circuit", "total_race_laps", "season",
    # Driver form
    "driver_rolling_avg_3", "driver_rolling_avg_5", "driver_dnf_rate_season", "driver_encoded",
    # Constructor form
    "constructor_rolling_avg_points_3", "constructor_season_points", "team_encoded",
    # Strategy
    "num_pit_stops", "compound_encoded",
    # Race conditions
    "is_wet_race", "safety_car_deployed", "vsc_deployed", "red_flag",
    "avg_air_temp", "avg_track_temp",
    # Pace
    "median_lap_time", "pace_delta",
    # Auto-added (empirically validated, leakage-checked)
    "clean_pace_delta", "teammate_pace_delta", "first_stint_len",
]
TARGET_COL = "final_position"
with open(MODELS_DIR / "feature_list.json", "w") as f:
    json.dump(FEATURE_COLS, f, indent=2)
print(f"Saved models/feature_list.json ({len(FEATURE_COLS)} features). Target: {TARGET_COL}")

Saved models/feature_list.json (24 features). Target: final_position


## Step 8 — Train/Test Split (by season, never random)

In [30]:
train_df = race_df[race_df["season"].isin([2022, 2023, 2024])].copy()
test_df = race_df[race_df["season"] == 2025].copy()
print(f'Train: {len(train_df)} rows ({train_df["season"].nunique()} seasons)')
print(f'Test : {len(test_df)} rows ({test_df["round"].nunique()} races)')

race_df.to_csv(PROC_DIR / "race_features.csv", index=False)
train_df.to_csv(PROC_DIR / "train.csv", index=False)
test_df.to_csv(PROC_DIR / "test.csv", index=False)
print("Saved race_features.csv / train.csv / test.csv to data/processed/")

Train: 1350 rows (3 seasons)
Test : 476 rows (24 races)
Saved race_features.csv / train.csv / test.csv to data/processed/


## Step 10 — Pre-Race (Forecasting) Feature Set
A parallel, **strictly pre-race** view for the prediction-vs-explanation comparison: only features
knowable before lights-out (grid, prior-race form, circuit, declared starting tyre, weather forecast).
All in-race pace/strategy/incident features are excluded. Saved to `data/processed/pre_race/` and
modelled in `notebooks/03_1_model_training.ipynb`. The full reconstruction set above is unchanged.

In [31]:
# Pre-race feature set: only quantities knowable BEFORE lights-out (no in-race pace/strategy/incidents).
# Parallel "forecasting" track to the full "reconstruction" set above; modelled in notebook 03_1.
PRE_RACE_FEATURES = [
    "grid_position", "season", "total_race_laps", "is_street_circuit",
    "driver_rolling_avg_3", "driver_rolling_avg_5", "driver_dnf_rate_season", "driver_encoded",
    "constructor_rolling_avg_points_3", "constructor_season_points", "team_encoded",
    "compound_encoded",   # declared starting tyre (known pre-race)
    "is_wet_race",        # forecast proxy (whether it actually rained is during-race; assumption noted)
]
META_COLS = ["season", "round", "driver_abbrev", "team", "dnf"]
keep = META_COLS + [c for c in PRE_RACE_FEATURES if c not in META_COLS] + [TARGET_COL]

pre_df = race_df[keep].copy()
PRE_DIR = PROC_DIR / "pre_race"; PRE_DIR.mkdir(parents=True, exist_ok=True)
pre_df.to_csv(PRE_DIR / "pre_race_features.csv", index=False)
pre_df[pre_df["season"].isin([2022, 2023, 2024])].to_csv(PRE_DIR / "pre_race_train.csv", index=False)
pre_df[pre_df["season"] == 2025].to_csv(PRE_DIR / "pre_race_test.csv", index=False)
with open(MODELS_DIR / "pre_race_feature_list.json", "w") as f:
    json.dump(PRE_RACE_FEATURES, f, indent=2)

print(f"Pre-race (forecasting) set: {len(PRE_RACE_FEATURES)} features -> data/processed/pre_race/")
print("  saved pre_race_features.csv / pre_race_train.csv / pre_race_test.csv")
print("  saved models/pre_race_feature_list.json")
print("Excluded (in-race only): num_pit_stops, safety_car_deployed, vsc_deployed, red_flag,")
print("  avg_air_temp, avg_track_temp, median_lap_time, pace_delta, clean_pace_delta,")
print("  teammate_pace_delta, first_stint_len")

Pre-race (forecasting) set: 13 features -> data/processed/pre_race/
  saved pre_race_features.csv / pre_race_train.csv / pre_race_test.csv
  saved models/pre_race_feature_list.json
Excluded (in-race only): num_pit_stops, safety_car_deployed, vsc_deployed, red_flag,
  avg_air_temp, avg_track_temp, median_lap_time, pace_delta, clean_pace_delta,
  teammate_pace_delta, first_stint_len


## Step 9 — Validation Checks

In [32]:
print("=== FEATURE NULLS ===")
print(race_df[FEATURE_COLS].isnull().sum().to_string())

print("\n=== TARGET DISTRIBUTION ===")
print(race_df["final_position"].describe().to_string())

print("\n=== TRAIN/TEST SHAPE ===")
print("Train:", train_df.shape, "| Test:", test_df.shape)

print("\n=== ROLLING FEATURE SAMPLE (VER, first 10) ===")
ver = race_df[race_df["driver"] == "VER"][
    ["season", "round", "event_name", "final_position",
     "driver_rolling_avg_3", "driver_rolling_avg_5"]].head(10)
print(ver.to_string(index=False))
print("(row 1 of 2022 -> NaN rolling; row 2 -> 1-race window. Confirms shift(1) is leak-free.)")

=== FEATURE NULLS ===
grid_position                        0
is_street_circuit                    0
total_race_laps                      0
season                               0
driver_rolling_avg_3                31
driver_rolling_avg_5                31
driver_dnf_rate_season              89
driver_encoded                       0
constructor_rolling_avg_points_3    20
constructor_season_points           80
team_encoded                         0
num_pit_stops                        0
compound_encoded                     0
is_wet_race                          0
safety_car_deployed                  0
vsc_deployed                         0
red_flag                             0
avg_air_temp                         0
avg_track_temp                       0
median_lap_time                     50
pace_delta                          50
clean_pace_delta                    50
teammate_pace_delta                  0
first_stint_len                      0

=== TARGET DISTRIBUTION ===
count    1826

In [33]:
# Explicit constructor-leakage spot-check: a team's season_points before its 2nd race must equal
# ONLY its 1st-race points (both drivers), never include the current race.
chk_team = train_df["team"].iloc[0]
ex = (race_df[(race_df.season == 2022) & (race_df.team == chk_team)]
      .sort_values("round")[["round", "driver", "points_scored",
                             "constructor_season_points", "constructor_rolling_avg_points_3"]])
print(f"Constructor leakage spot-check for team '{chk_team}' (2022, by round):")
print(ex.head(6).to_string(index=False))
print("Round 1 rows -> constructor_season_points should be NaN (no prior race) = leak-free.")

Constructor leakage spot-check for team 'Williams' (2022, by round):
 round driver  points_scored  constructor_season_points  constructor_rolling_avg_points_3
     1    ALB            0.0                        NaN                               NaN
     1    LAT            0.0                        NaN                               NaN
     2    ALB            0.0                        0.0                               0.0
     2    LAT            0.0                        0.0                               0.0
     3    ALB            1.0                        0.0                               0.0
     3    LAT            0.0                        0.0                               0.0
Round 1 rows -> constructor_season_points should be NaN (no prior race) = leak-free.


## CP2 Final Summary

In [34]:
null_counts = race_df[FEATURE_COLS].isnull().sum()
nulls_present = null_counts[null_counts > 0]
n_no_pace = int(race_df["median_lap_time"].isna().sum())

print("=== CP2 COMPLETE ===")
print(f"Race-level rows: {len(race_df)}")
print(f"Train rows: {len(train_df)} (2022-2024)")
print(f"Test rows: {len(test_df)} (2025)")
print(f"Features: {len(FEATURE_COLS)} columns (21 base + 3 auto-added)")
print("Nulls in feature matrix:",
      "none" if nulls_present.empty else "\n  " + nulls_present.to_string().replace("\n", "\n  "))
print("Files saved:")
for p in ["data/processed/race_features.csv", "data/processed/train.csv",
          "data/processed/test.csv", "models/feature_list.json", "models/encoders.pkl"]:
    print("  -", p)
print("Rolling feature check (VER first 5 races):")
print(race_df[race_df.driver == "VER"][["season","round","final_position",
      "driver_rolling_avg_3","driver_rolling_avg_5"]].head(5).to_string(index=False))
print("Known issues for CP3:")
print(f"  - Early-career races: NaN driver rolling form (expected; min_periods=1 minimises)")
print(f"  - {n_no_pace} driver-races have no accurate laps -> median_lap_time/pace_delta/clean_pace_delta NaN")
print(f"  - stint nulls in raw laps -> minor stint_lap/strategy gaps")
print(f"  - 'season' is a feature but test=2025 is an unseen value (tree-safe; reconsider in CP3)")
print(f"  - 2025 rookies get encoded ids unseen in training (modeling concern, not leakage)")
print(f"  - Rejected low-signal features (kept out): tyre-deg slope, laptime CV, lap-1/3 start gain")

=== CP2 COMPLETE ===
Race-level rows: 1826
Train rows: 1350 (2022-2024)
Test rows: 476 (2025)
Features: 24 columns (21 base + 3 auto-added)
Nulls in feature matrix: 
  driver_rolling_avg_3                31
  driver_rolling_avg_5                31
  driver_dnf_rate_season              89
  constructor_rolling_avg_points_3    20
  constructor_season_points           80
  median_lap_time                     50
  pace_delta                          50
  clean_pace_delta                    50
Files saved:
  - data/processed/race_features.csv
  - data/processed/train.csv
  - data/processed/test.csv
  - models/feature_list.json
  - models/encoders.pkl
Rolling feature check (VER first 5 races):
 season  round  final_position  driver_rolling_avg_3  driver_rolling_avg_5
   2022      1            19.0                   NaN                   NaN
   2022      2             1.0             19.000000             19.000000
   2022      3            18.0             10.000000             10.000000
   